# dexcup: train the teacher on a cloud GPU

Works on Google Colab or Kaggle (free GPU tiers are enough). Steps:
1. **Runtime -> Change runtime type -> GPU** (Colab) or **Settings -> Accelerator -> GPU** (Kaggle).
2. Run the cells in order. Cell 2 asks you to upload `dexcup_cloud.zip` from the project folder.
3. The last cell downloads `teacher.pt`. Put it in the project's `models/` folder on the laptop.

Training needs only PyTorch + numpy (preinstalled on both). No MuJoCo here: the sim evaluation runs on the laptop.

In [ ]:
# 1. Check the GPU is attached (should print a GPU name, e.g. Tesla T4)
import torch
print('CUDA available:', torch.cuda.is_available(), '|', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NO GPU: change the runtime type first')

In [ ]:
# 2. Upload dexcup_cloud.zip and unpack it
import os, zipfile
try:
    from google.colab import files          # Colab: opens a file picker
    up = files.upload()
    zpath = next(iter(up))
except ImportError:                          # Kaggle: add the zip as a dataset, then set its path here
    zpath = '/kaggle/input/dexcup-cloud/dexcup_cloud.zip'
zipfile.ZipFile(zpath).extractall('.')
os.chdir('dexcup')
print(os.listdir('.'), len(os.listdir('data/replayed')), 'episodes')

In [ ]:
# 3. Quick check: 5 batches (should finish in seconds on a GPU)
!python train_teacher.py --epochs 1 --max-steps 5

In [ ]:
# 4. Full training: 80 epochs. Progress prints every epoch; the best model (lowest val loss) is kept.
!python train_teacher.py --epochs 80

In [ ]:
# 5. Download the trained teacher -> put it in the project's models/ folder on the laptop
try:
    from google.colab import files
    files.download('models/teacher.pt')
except ImportError:
    import shutil; shutil.copy('models/teacher.pt', '/kaggle/working/teacher.pt')
    print('Kaggle: download teacher.pt from the Output panel on the right')